# Docling as a Tool for AI Agents via MCP
*Exposing document processing capabilities through the Model Context Protocol*

## Lab 7: Docling + MCP for AI Agents

In this lab, you'll learn how to expose Docling's document processing capabilities to AI agents using the **Model Context Protocol (MCP)**. This enables:

- **AI-Native Integration**: LLMs can directly invoke Docling tools during conversations
- **Agentic Document Processing**: AI agents can convert, create, and export documents autonomously
- **Fully Open-Source Stack**: docling-mcp + Continue + Granite via Ollama
- **Zero Custom Code**: Use the official docling-mcp package — no MCP server coding required

### What is MCP?

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open standard that allows AI models to discover and use external tools. Instead of calling APIs directly, the LLM can invoke tools through a standardized protocol, making AI agents more capable and extensible.

### How This Connects to Lab 5

In Lab 5, you deployed Docling as a REST API (docling-serve) for programmatic access. MCP takes this further: instead of *you* writing code to call the API, the *AI agent* discovers and calls Docling tools autonomously. This is the modern pattern for integrating capabilities into AI workflows.

## Prerequisites

Before we begin, ensure you have:
- Python 3.11 or later installed
- [VS Code](https://code.visualstudio.com/) installed
- [Ollama](https://ollama.ai/) installed and running
- docling-serve from Lab 5 running in a terminal (`docling-serve run`). docling-mcp sends its document conversions to it.
- Completed Labs 1-5 (recommended)
- Basic familiarity with VS Code extensions

> **Note**: This lab is **local-only**. MCP requires a local server process and VS Code extension configuration, which cannot run on Google Colab.

## Setting up the environment

Ensure you are running Python 3.11 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 11) and sys.version_info < (3, 14), "Use Python 3.11, 3.12, or 3.13 to run this notebook."

## Install Dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install docling-mcp mcp httpx
! echo "::endgroup::"

## Architecture Overview

This lab uses four components. Continue and docling-mcp talk to each other through MCP:

```text
+------------------+       MCP (stdio)       +------------------+
|                  | <---------------------> |                  |
|   Continue       |                         |   docling-mcp    |
|   (VS Code)      |                         |   server         |
|                  |                         |                  |
+--------+---------+                         +--------+---------+
         |                                          |
         | LLM calls                                | HTTP
         v                                          v
+------------------+                         +------------------+
|   Granite 4.2    |                         |   docling-serve  |
|   via Ollama     |                         |   (Lab 5)        |
+------------------+                         +------------------+
```

1. **Continue** (MCP Client): Open-source VS Code extension that acts as an AI coding assistant
2. **docling-mcp** (MCP Server): Exposes Docling tools via the MCP protocol
3. **docling-serve** (Docling service): Runs the actual document conversions for docling-mcp
4. **Granite via Ollama** (LLM): The language model that decides when to use tools

When you ask Continue to process a document, Granite recognizes that it needs Docling's capabilities and invokes the appropriate MCP tools automatically.

## Understanding docling-mcp Tools

The [`docling-mcp`](https://github.com/docling-project/docling-mcp) package exposes three categories of tools:

### Conversion Tools

| Tool | Description |
| :--- | :--- |
| `convert_document_into_docling_document` | Convert a document (URL or local path) and return its document key |
| `convert_directory_files_into_docling_document` | Convert all files from a local directory |
| `check_document_in_local_cache` | Check if a document is already converted and cached |
| `list_cached_documents` | List the keys of all cached documents |
| `remove_document_from_local_cache` | Remove a document from the cache |

### Document Generation Tools

| Tool | Description |
| :--- | :--- |
| `create_new_docling_document` | Create a new, empty DoclingDocument |
| `add_title_to_docling_document` | Add or update the title |
| `add_section_heading_to_docling_document` | Add a section heading |
| `add_paragraph_to_docling_document` | Add a paragraph |
| `open_list_in_docling_document` | Open a list structure |
| `add_list_items_to_list_in_docling_document` | Add items to an open list |
| `close_list_in_docling_document` | Close the current list |
| `add_table_in_html_format_to_docling_document` | Add an HTML-formatted table |
| `export_docling_document_to_markdown` | Export a cached document (converted or generated) as Markdown |
| `save_docling_document` | Save to disk in Markdown and JSON formats |
| `get_page_thumbnail` | Generate a thumbnail image for a page |

### Document Manipulation Tools

| Tool | Description |
| :--- | :--- |
| `get_overview_of_document_anchors` | Get a structured overview of a document |
| `search_for_text_in_document_anchors` | Search for text within document anchors |
| `get_text_of_document_item_at_anchor` | Retrieve text at a specific anchor |
| `update_text_of_document_item_at_anchor` | Update text at a specific anchor |
| `delete_document_items_at_anchors` | Delete items at specified anchors |

These tools allow an AI agent to **consume** existing documents, **create** new ones, and **manipulate** document content.

## Step 1: Set Up Ollama with Granite

First, ensure Ollama is installed and pull a Granite model that supports tool use.

### Install Ollama

Download from [ollama.ai](https://ollama.ai/) if not already installed.

### Pull a Granite Model

Open a terminal and run:

```bash
ollama pull granite4.2:8b
```

This pulls IBM Granite 4.2 (8B parameters), which supports tool calling — a requirement for MCP to work.

In [ ]:
import urllib.request
import urllib.error
import json

def check_ollama():
    """Check if Ollama is running and list available models."""
    try:
        req = urllib.request.Request("http://localhost:11434/api/tags", method="GET")
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            models = [m["name"] for m in data.get("models", [])]
            print("Ollama is running!")
            print(f"Available models: {models}")

            granite_models = [m for m in models if "granite" in m.lower()]
            if granite_models:
                print(f"\nGranite models found: {granite_models}")
            else:
                print("\nNo Granite models found. Please run:")
                print("  ollama pull granite4.2:8b")
            return True
    except (urllib.error.URLError, TimeoutError):
        print("Ollama is not running. Please start Ollama first.")
        print("  On macOS: Open the Ollama app")
        print("  On Linux: ollama serve")
        return False

ollama_available = check_ollama()

## Step 2: Verify docling-mcp Server

Let's verify that the docling-mcp server can be launched. The server will be started automatically by Continue, but we can test it independently first.

docling-mcp does not convert documents itself by default: it sends them to the Docling service at `DOCLING_MCP_SERVICE_URL`. In this lab that is the docling-serve instance from Lab 5, so start it in a separate terminal and leave it running:

```bash
docling-serve run
```

### Quick Launch Test

In another terminal, run:

```bash
DOCLING_MCP_SERVICE_URL=http://localhost:5001 uvx --from docling-mcp docling-mcp-server --transport streamable-http
```

This starts the server over HTTP on port 8000 (for manual testing). You should see output indicating the server is ready. Press `Ctrl+C` to stop it when done.

> **Note**: Continue launches the server itself over the `stdio` transport, so you do not need to start it manually. Since streamable HTTP is the default transport, the Continue configuration below passes `--transport=stdio` explicitly.

In [ ]:
import subprocess
import shutil

# Check if uvx is available
uvx_path = shutil.which("uvx")
if uvx_path:
    print(f"uvx found at: {uvx_path}")
    # Verify docling-mcp can be resolved
    result = subprocess.run(
        ["uvx", "--from", "docling-mcp", "docling-mcp-server", "--help"],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        print("docling-mcp server is available!")
        print("\nServer help output:")
        print(result.stdout[:500])
    else:
        print("Error launching docling-mcp:")
        print(result.stderr[:500])
else:
    print("uvx not found. Install it with: pip install uv")

In [ ]:
import urllib.error
import urllib.request

# Check that docling-serve (from Lab 5) is reachable; docling-mcp sends conversions to it
DOCLING_SERVE_URL = "http://localhost:5001"

try:
    with urllib.request.urlopen(f"{DOCLING_SERVE_URL}/health", timeout=5) as response:
        print(f"docling-serve is running at {DOCLING_SERVE_URL}")
except (urllib.error.URLError, TimeoutError):
    print(f"docling-serve is not reachable at {DOCLING_SERVE_URL}. Start it in a terminal with:")
    print("  docling-serve run")

## Step 3: Install and Configure Continue

### Install the Continue Extension

1. Open VS Code
2. Go to Extensions (`Ctrl+Shift+X` / `Cmd+Shift+X`)
3. Search for "Continue"
4. Install the **Continue** extension by [continue.dev](https://continue.dev)

### Configure Ollama as the Model Provider

After installation, Continue needs to be configured to use your local Granite model. Edit your Continue configuration file:

**Location**: `~/.continue/config.yaml`

Add or modify the models section:

```yaml
name: Main Config
version: 1.0.0
schema: v1

models:
  - name: Granite 4.2
    provider: ollama
    model: granite4.2:8b
    requestOptions:
      extraBodyProperties:
        options:
          num_ctx: 16384
    apiBase: http://localhost:11434
    roles:
      - chat
      - edit
      - apply
    capabilities:
      - tool_use
```

### Configure the docling-mcp Server

Add the MCP server configuration to the same `config.yaml`:

```yaml
mcpServers:
  - name: Docling
    command: uvx
    args:
      - --from=docling-mcp
      - docling-mcp-server
      - --transport=stdio
    env:
      DOCLING_MCP_SERVICE_URL: http://localhost:5001
```

`--transport=stdio` lets Continue talk to the server over its standard input and output, and `DOCLING_MCP_SERVICE_URL` points docling-mcp at your docling-serve instance.

<details>
<summary>Using a hosted Docling service instead of docling-serve?</summary>

Point docling-mcp at the service and pass your API key. This works with [Docling for IBM watsonx](https://www.ibm.com/products/docling) or any docling-serve that requires an API key.

```yaml
    env:
      DOCLING_MCP_SERVICE_URL: <your-docling-service-url>
      DOCLING_MCP_SERVICE_API_KEY: <your-api-key>
```

</details>

<details>
<summary>Prefer to convert inside docling-mcp instead of docling-serve?</summary>

Install the local conversion extra and switch the conversion mode. The first launch downloads the full Docling package and its models, so it takes several minutes.

```yaml
    args:
      - --from=docling-mcp[local]
      - docling-mcp-server
      - --transport=stdio
    env:
      DOCLING_MCP_CONVERSION_MODE: local
```

</details>

### Complete config.yaml

Here is the full configuration file:

```yaml
name: Main Config
version: 1.0.0
schema: v1

models:
  - name: Granite 4.2
    provider: ollama
    model: granite4.2:8b
    requestOptions:
      extraBodyProperties:
        options:
          num_ctx: 16384
    apiBase: http://localhost:11434
    roles:
      - chat
      - edit
      - apply
    capabilities:
      - tool_use

mcpServers:
  - name: Docling
    command: uvx
    args:
      - --from=docling-mcp
      - docling-mcp-server
      - --transport=stdio
    env:
      DOCLING_MCP_SERVICE_URL: http://localhost:5001
```

### Verify in Continue

1. Open Continue in VS Code (click the Continue icon in the sidebar)
2. Switch to **Agent** mode (MCP tools are only available in Agent mode)
3. You should see the Granite model available in the model selector
4. The Docling MCP tools should appear in the tools panel

In [ ]:
from pathlib import Path

config_path = Path.home() / ".continue" / "config.yaml"
if config_path.exists():
    print(f"Continue config found at: {config_path}")
    print("\nCurrent contents:")
    print(config_path.read_text()[:1000])
else:
    print(f"Continue config not found at: {config_path}")
    print("This file will be created when you install and configure Continue.")
    print("See the instructions above for the recommended configuration.")

## Step 4: Using Docling Tools Through Continue

Now that everything is configured, let's use the Docling MCP tools through Continue.

> **Important**: Switch Continue to **Agent** mode. MCP tools are only available in Agent mode, not in Chat mode.

### Exercise 1: Convert a Document

In Continue's Agent mode, type the following prompts, one at a time:

```text
Use Docling to convert https://arxiv.org/pdf/2501.17887 and tell me the document key.
```

```text
Use Docling to save that document, and tell me the file paths it was saved to.
```

```text
Export that document as markdown with a max_size of 6000 characters, then give me a concise 3-bullet executive summary.
```

What happens behind the scenes:
1. Granite receives your first prompt and recognizes it needs document conversion
2. It invokes the `convert_document_into_docling_document` MCP tool with the URL
3. docling-mcp sends the PDF to docling-serve, caches the converted document, and returns its document key
4. For the second prompt, Granite calls `save_docling_document` with that key, which writes Markdown and JSON files to the docling-mcp cache directory and returns their paths
5. For the third prompt, Granite calls `export_docling_document_to_markdown` with that key and `max_size`, which keeps the text small enough for the model's context window, and summarizes it for you in the chat

### Exercise 2: Create a New Document

Creating a document takes several tool calls in a strict order. Small models like Granite 4.2 can lose track when you ask for everything in one prompt, and may invent tool names that do not exist. Instead, build the document one step at a time, naming the Docling action in each prompt. Run these prompts one at a time in the same chat:

```text
Use Docling to create a new document with the prompt "Meeting Notes" and tell me the document key.
```

```text
Use Docling to add the title "Meeting Notes" to that document.
```

```text
Use Docling to add the section heading "Action Items" at level 1 to that document.
```

```text
Use Docling to open a list in that document.
```

```text
Use Docling to add these items to the open list, using "-" as the marker: "Review PR #42", "Update documentation", "Schedule follow-up meeting".
```

```text
Use Docling to close the list in that document.
```

```text
Use Docling to export that document as markdown.
```

The tools invoked, in order, are:
1. `create_new_docling_document`
2. `add_title_to_docling_document`
3. `add_section_heading_to_docling_document`
4. `open_list_in_docling_document`
5. `add_list_items_to_list_in_docling_document`
6. `close_list_in_docling_document`
7. `export_docling_document_to_markdown`

> **Note**: A list must be open before you add items, and closed before you add a title, heading, or paragraph. If a step fails, the error message tells the agent what to fix.

### Exercise 3: Search a Converted Document

Reading a whole paper into the chat can fill a small model's context window. The manipulation tools let the agent look at just the parts it needs. This exercise uses the paper you converted in Exercise 1, which is still in the server's cache as long as the docling-mcp server is running. If Continue does not remember the document key, ask it to list the cached documents first.

```text
Use Docling to list the cached documents.
```

```text
Use Docling to search for the text "table" in that document and show me the anchors.
```

```text
Use Docling to get the text of the first anchor from that search.
```

The tools invoked, in order, are:
1. `list_cached_documents`
2. `search_for_text_in_document_anchors`
3. `get_text_of_document_item_at_anchor`

## Step 5: Testing the MCP Server Programmatically (Optional)

While the primary workflow is through Continue, you can also test the docling-mcp server programmatically using the MCP Python SDK. This is useful for debugging or building custom integrations.

In [ ]:
try:
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    MCP_SDK_AVAILABLE = True
    print("MCP Python SDK is available")
except ImportError:
    MCP_SDK_AVAILABLE = False
    print("MCP Python SDK not installed. This is optional.")
    print("Install with: uv pip install mcp")

In [ ]:
if MCP_SDK_AVAILABLE:
    # Launch docling-mcp the same way Continue does: over stdio, pointed at docling-serve
    server_params = StdioServerParameters(
        command="uvx",
        args=["--from=docling-mcp", "docling-mcp-server", "--transport=stdio"],
        env={"DOCLING_MCP_SERVICE_URL": DOCLING_SERVE_URL},
    )

    async def list_docling_tools():
        """Connect to docling-mcp and list available tools."""
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()

                print(f"docling-mcp exposes {len(tools.tools)} tools:\n")
                for tool in tools.tools:
                    print(f"  - {tool.name}")
                    if tool.description:
                        print(f"    {tool.description[:80]}")
                    print()

    await list_docling_tools()
else:
    print("Skipping -- MCP SDK not installed")

In [ ]:
import json

if MCP_SDK_AVAILABLE:
    async def test_convert_document():
        """Test document conversion via MCP."""
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                # Call the convert tool; it returns a key for the cached document
                print("Converting document via MCP...")
                result = await session.call_tool(
                    "convert_document_into_docling_document",
                    arguments={"source": "https://arxiv.org/pdf/2501.17887"}
                )
                document_key = json.loads(result.content[0].text)["document_key"]
                print(f"Document key: {document_key}")

                # Export the cached document to Markdown, as an agent would
                result = await session.call_tool(
                    "export_docling_document_to_markdown",
                    arguments={"document_key": document_key}
                )
                markdown = json.loads(result.content[0].text)["markdown"]
                print(f"\nMarkdown preview ({len(markdown)} chars):")
                print(markdown[:500])
                print("...")

    await test_convert_document()
else:
    print("Skipping -- MCP SDK not installed")

## MCP vs REST API: When to Use What

You've now seen two ways to expose Docling as a service:

| Feature | docling-serve (Lab 5) | docling-mcp (Lab 7) |
| :--- | :--- | :--- |
| **Protocol** | REST/HTTP | MCP (stdio/sse/http) |
| **Primary Consumer** | Your application code | AI agents/LLMs |
| **Discovery** | Read API docs manually | Tools auto-discovered by LLM |
| **Invocation** | You write HTTP requests | LLM decides when to call tools |
| **Use Case** | Microservices, batch processing | AI-assisted workflows, copilots |
| **Scaling** | Load balancers, containers | Per-agent instance |
| **Best For** | Production pipelines | Interactive AI experiences |

### The Key Difference

With **docling-serve**, you (the developer) decide when to call the API. With **docling-mcp**, the AI agent decides when to invoke Docling tools based on the user's natural language request.

## Troubleshooting

### Ollama Issues

| Problem | Solution |
| :--- | :--- |
| "Ollama is not running" | Start the Ollama app (macOS) or run `ollama serve` (Linux) |
| Model not found | Run `ollama pull granite4.2:8b` |
| Slow responses | Try a smaller model: `ollama pull granite4.2:3b` |

### Continue Issues

| Problem | Solution |
| :--- | :--- |
| MCP tools not showing | Ensure you are in **Agent** mode, not Chat mode |
| "Cannot connect to MCP server" | Verify `uvx` is on your PATH: `which uvx` |
| Model not appearing | Check `config.yaml` syntax and restart VS Code |
| Tool calls failing | Check Continue output panel for error logs |

### docling-mcp Issues

| Problem | Solution |
| :--- | :--- |
| "uvx not found" | Install uv: `pip install uv` or `brew install uv` |
| Import errors | Run `uvx --from docling-mcp docling-mcp-server --help` to debug |
| Conversion timeout | Large documents may take time; try a smaller PDF first |
| Conversion fails with a remote configuration error | Start docling-serve (`docling-serve run`) and set `DOCLING_MCP_SERVICE_URL` to its address |
| Continue lists the server but never shows its tools | Make sure the `args` include `--transport=stdio` |

## Summary

In this lab, you learned how to:

1. **Set up the MCP stack**: Ollama (Granite) + Continue + docling-mcp
2. **Configure MCP tools**: Connect docling-mcp to Continue via config.yaml
3. **Use Docling through AI agents**: Convert and create documents via natural language
4. **Compare approaches**: When to use REST API (Lab 5) vs MCP (Lab 7)

### The AI Agent Paradigm

MCP represents a shift in how we integrate tools with AI:
- **Before MCP**: Developers write code to call APIs
- **With MCP**: AI agents discover and invoke tools autonomously

Docling's MCP integration means document processing becomes a natural capability of any AI agent, accessible through conversation rather than code.

### Resources

- [docling-mcp GitHub](https://github.com/docling-project/docling-mcp)
- [Model Context Protocol Specification](https://modelcontextprotocol.io/)
- [Continue Documentation](https://docs.continue.dev/)
- [Ollama](https://ollama.ai/)
- [Docling Documentation](https://docling-project.github.io/docling/)
- [IBM Granite Models](https://www.ibm.com/granite)